# 04 — Baseline Embedding Retrieval Evaluation

This notebook establishes the **baseline retrieval performance** of the original
embedding model before any domain-specific fine-tuning.

This step is essential. Fine-tuning is only useful if we can later demonstrate that
the adapted model performs better than the original model under the **same evaluation
conditions**.

## Where this notebook fits in the pipeline

```text
Validated legal articles
        ↓
03 — Dataset construction
        ↓
Passages + queries + qrels
        ↓
04 — Baseline evaluation       ← this notebook
        ↓
Baseline retrieval metrics
        ↓
05 — Hard-negative mining
        ↓
06 — Fine-tuning
        ↓
Final comparison
```

## What is a baseline?

A baseline is a reference result obtained **before the proposed improvement is
applied**.

Here the baseline model is:

```text
BAAI/bge-m3
```

without legal-domain fine-tuning.

Later we can compare:

```text
Original BGE-M3
      vs.
Fine-tuned BGE-M3
```

using exactly the same passages, queries, relevance judgements and metrics.

## Retrieval evaluation

For each query:

```text
Query text
    ↓
Embedding model
    ↓
Query vector
    ↓
Similarity against every passage vector
    ↓
Ranked list of passages
    ↓
Compare ranking with qrels
```

The evaluation uses:

- **Recall@1, Recall@5 and Recall@10**;
- **MRR@10**;
- **nDCG@10**.

Results are also separated into **reference** and **semantic** queries.

## Learning objectives

After this notebook, you should understand:

- what an embedding is in the context of retrieval;
- why query and passage vectors must be produced by the same model;
- why vector normalisation allows cosine similarity to be computed efficiently;
- how top-K retrieval works;
- how Recall, MRR and nDCG evaluate different aspects of ranking;
- why semantic and reference queries should be analysed separately;
- why failure analysis is as important as aggregate metrics.

## 1. Install dependencies if necessary

The retrieval experiment uses Sentence Transformers to load and run the embedding
model, PyTorch for model execution, NumPy for vector operations and pandas for result
analysis.

Package installation should normally be done once for the project environment. If the
packages are already installed, this cell can be skipped.

After major package upgrades, restarting the Jupyter kernel is generally advisable so
the notebook uses the newly installed versions.

In [3]:
# Uncomment only if the packages are not already installed.
%pip install -U sentence-transformers torch pandas numpy tqdm

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/611.3 kB ? eta -:--:--
   ---------------------------------------- 611.3/611.3 kB 6.9 MB/s  0:00:00
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/11.7 MB 8.7 MB/s eta 0:00:02
   ------------- -------------------------- 3.9/11.7 MB 9.5 MB/s eta 0:00:01
   --------------------- ------------------ 6.3/11.7 MB 10.3 MB/s eta 0:00:01
   ------------------------------- -------- 9.2/11.7 MB 11.0 MB/s eta 0:00:01
   ---------------------------------------  11.5/11.7 MB 11.5 MB/s eta 0:00:01
   ---------------------------------------- 11.7/11.7 MB 11.2 MB/s  0:00:01
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 13.9 MB/s  0:00:00
   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   - --------------------

  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports and project paths

This notebook consumes the dataset created in notebook 03 and writes evaluation
artifacts to `results/baseline/`.

The key input is `passages.jsonl`, which represents the complete searchable legal
corpus. Validation and test queries/qrels are loaded later.

Keeping results outside the dataset directory is deliberate:

```text
data/       → experimental inputs
results/    → experimental outputs
```

This separation makes it easier to rerun evaluation without modifying the source
dataset.

In [4]:
from pathlib import Path
from collections import defaultdict
import json
import math
import time

import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

PROJECT_ROOT = Path("..")
DATASET_DIR = PROJECT_ROOT / "data" / "dataset_v2"
RESULTS_DIR = PROJECT_ROOT / "results" / "baseline"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PASSAGES_PATH = DATASET_DIR / "passages.jsonl"

print(f"Dataset : {DATASET_DIR.resolve()}")
print(f"Results : {RESULTS_DIR.resolve()}")

C:\Users\user\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset : C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\data\dataset_v2
Results : C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\baseline


## 3. Baseline model configuration

`BAAI/bge-m3` is evaluated exactly as it exists before fine-tuning.

Important parameters are:

- `BATCH_SIZE` — how many texts are encoded together;
- `MAX_SEQ_LENGTH` — maximum token sequence accepted during encoding;
- `TOP_K` — number of highest-ranked passages retained per query;
- `DEVICE` — GPU when CUDA is available, otherwise CPU.

### Batch size and memory

A larger batch can improve throughput but consumes more GPU memory. It does **not**
change the conceptual retrieval task.

### Why evaluate validation and test?

The validation set is useful during development and later checkpoint selection. The
test set provides the final held-out estimate.

The training split is intentionally absent from baseline reporting because our main
question is how the model generalises to articles outside the training subset.

In [5]:
MODEL_NAME = "BAAI/bge-m3"

BATCH_SIZE = 32
MAX_SEQ_LENGTH = 512

TOP_K = 10

EVALUATION_SPLITS = [
    "validation",
    "test",
]

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(f"Model  : {MODEL_NAME}")
print(f"Device : {DEVICE}")

if DEVICE == "cuda":
    print(
        "GPU    :",
        torch.cuda.get_device_name(0)
    )

Model  : BAAI/bge-m3
Device : cpu


## 4. JSONL loader

The dataset uses JSON Lines, where each line is an independent JSON record.

This helper reads the file line by line and reports the exact line number if malformed
JSON is encountered.

Explicit validation errors are preferable to silently ignoring damaged records,
especially in an evaluation pipeline where missing data could change the reported
metrics.

In [6]:
def read_jsonl(path):
    records = []

    with path.open(
        "r",
        encoding="utf-8"
    ) as f:
        for line_number, line in enumerate(
            f,
            start=1
        ):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(
                    json.loads(line)
                )
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {path.name}, "
                    f"line {line_number}: {exc}"
                )

    return records

## 5. Load passages, queries and qrels

Three objects are required for retrieval evaluation:

```text
PASSAGES
the searchable legal corpus

QUERIES
the questions submitted to the embedding model

QRELS
the ground-truth query → relevant passage relationships
```

The passage corpus is shared across evaluation splits. Queries and qrels are loaded
separately for validation and test.

The printed counts provide the first consistency check against the dataset report from
notebook 03.

In [7]:
passages = read_jsonl(
    PASSAGES_PATH
)

queries_by_split = {}
qrels_by_split = {}

for split in EVALUATION_SPLITS:
    queries_by_split[split] = read_jsonl(
        DATASET_DIR
        / f"{split}_queries.jsonl"
    )

    qrels_by_split[split] = read_jsonl(
        DATASET_DIR
        / f"{split}_qrels.jsonl"
    )

print(f"Passages: {len(passages)}")

for split in EVALUATION_SPLITS:
    print(
        f"{split:10} | "
        f"queries={len(queries_by_split[split]):5} | "
        f"qrels={len(qrels_by_split[split]):5}"
    )

Passages: 2088
validation | queries=  516 | qrels=  632
test       | queries=  561 | qrels=  953


## 6. Validate evaluation data

Before running an expensive embedding model, we verify the integrity of the evaluation
data.

The checks ensure that:

- passage IDs are unique;
- query IDs are unique inside each split;
- every qrel refers to an existing query;
- every qrel refers to an existing passage;
- every query has at least one relevant passage.

Without these guarantees, a low retrieval score might be caused by a broken dataset
rather than by the embedding model.

This is an example of a useful experimental principle:

> **Validate the ground truth before evaluating the model against it.**

In [8]:
passage_ids = [
    passage["passage_id"]
    for passage in passages
]

assert len(passage_ids) == len(
    set(passage_ids)
), "Duplicate passage IDs detected."

passage_id_set = set(passage_ids)

for split in EVALUATION_SPLITS:
    queries = queries_by_split[split]
    qrels = qrels_by_split[split]

    query_ids = [
        query["query_id"]
        for query in queries
    ]

    assert len(query_ids) == len(
        set(query_ids)
    ), f"Duplicate query IDs in {split}."

    query_id_set = set(query_ids)

    relevance = defaultdict(set)

    for qrel in qrels:
        assert (
            qrel["query_id"]
            in query_id_set
        ), (
            f"Unknown query in {split}: "
            f"{qrel['query_id']}"
        )

        assert (
            qrel["passage_id"]
            in passage_id_set
        ), (
            f"Unknown passage in {split}: "
            f"{qrel['passage_id']}"
        )

        if qrel.get("relevance", 0) > 0:
            relevance[
                qrel["query_id"]
            ].add(
                qrel["passage_id"]
            )

    missing = [
        query_id
        for query_id in query_ids
        if not relevance[query_id]
    ]

    assert not missing, (
        f"{len(missing)} queries without "
        f"relevant passages in {split}."
    )

print("Evaluation data validation passed.")

Evaluation data validation passed.


## 7. Load the original embedding model

`SentenceTransformer` loads the pretrained BGE-M3 model and places it on the selected
device.

An embedding model converts variable-length text into a fixed-length numerical vector:

```text
"Quais são as regras sobre localização das operações?"
                    ↓
                 BGE-M3
                    ↓
[0.012, -0.084, 0.031, ..., 0.067]
```

Texts with related meaning should ideally occupy nearby regions of the embedding
space.

The printed **embedding dimension** tells us how many numerical components each vector
contains.

At this stage the model is only used for inference. No weights are changed.

In [9]:
start = time.time()

model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)

model.max_seq_length = MAX_SEQ_LENGTH

print(
    f"Loaded in "
    f"{time.time() - start:.1f} s"
)
print(
    f"Embedding dimension: "
    f"{model.get_sentence_embedding_dimension()}"
)

C:\Users\user\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 15108.97it/s]


Loaded in 45.1 s
Embedding dimension: 1024


C:\Users\user\AppData\Local\Temp\ipykernel_23704\188830983.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"{model.get_sentence_embedding_dimension()}"


## 8. Encode the passage corpus

Every legal passage is converted into an embedding vector.

If there are `N` passages and each embedding has `D` dimensions, the resulting matrix
has shape:

```text
(N, D)
```

For example:

```text
1,600 passages × 1,024 dimensions
```

### Why `normalize_embeddings=True`?

Normalisation scales every embedding to unit length:

```text
||vector|| = 1
```

For unit-normalised vectors, the dot product is equivalent to cosine similarity:

```text
cosine_similarity(q, p) = q · p
```

This lets us rank passages efficiently using matrix multiplication later.

### Why encode passages only once?

The passage corpus is the same for every query. Re-encoding it for each query would be
wasteful. We therefore compute passage embeddings once and reuse them throughout the
evaluation.

In [10]:
passage_texts = [
    passage["text"]
    for passage in passages
]

start = time.time()

passage_embeddings = model.encode(
    passage_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

passage_embeddings = np.asarray(
    passage_embeddings,
    dtype=np.float32
)

print(
    "Passage embedding matrix:",
    passage_embeddings.shape
)

print(
    f"Encoding time: "
    f"{time.time() - start:.1f} s"
)
# The passage matrix is reused for every query. Its rows correspond exactly to
# the order of `passages`, which is why retrieved matrix indices can later be
# converted back into passage IDs.


Batches: 100%|██████████| 66/66 [33:22<00:00, 30.33s/it]

Passage embedding matrix: (2088, 1024)
Encoding time: 2002.2 s


## 9. Build relevance mappings

Qrels are stored as individual records, but metric functions need fast access to all
relevant passages for a given query.

We therefore transform:

```text
query A → passage 1
query A → passage 2
query B → passage 7
```

into:

```text
query A → {passage 1, passage 2}
query B → {passage 7}
```

A Python `set` is appropriate because relevance membership checks are frequent and
efficient.

In [11]:
def build_relevance_map(qrels):
    relevance = defaultdict(set)

    for item in qrels:
        if item.get("relevance", 0) > 0:
            relevance[
                item["query_id"]
            ].add(
                item["passage_id"]
            )

    return relevance


relevance_by_split = {
    split: build_relevance_map(
        qrels_by_split[split]
    )
    for split in EVALUATION_SPLITS
}

## 10. Retrieval metrics

No single metric describes every aspect of retrieval quality, so we calculate several.

### Recall@K

Recall asks:

> **What proportion of all relevant passages appeared in the first K results?**

```text
Recall@K =
relevant passages retrieved in top K
────────────────────────────────────
total relevant passages
```

If a query has two relevant chunks and the top five results contain one:

```text
Recall@5 = 1 / 2 = 0.50
```

### MRR@10 — Mean Reciprocal Rank

MRR focuses on the position of the **first relevant result**.

```text
relevant at rank 1 → reciprocal rank = 1
relevant at rank 2 → reciprocal rank = 1/2
relevant at rank 5 → reciprocal rank = 1/5
none in top 10     → 0
```

The final MRR is the mean across queries.

This is particularly useful when the user mainly needs one correct legal passage near
the top.

### nDCG@10

nDCG — Normalised Discounted Cumulative Gain — considers the ranking positions of
relevant results.

Relevant passages near the top contribute more than relevant passages near the bottom.

The score is normalised against the ideal ranking:

```text
0 ≤ nDCG ≤ 1
```

A value near 1 means the relevant passages are ranked close to their ideal positions.

### Why use several metrics?

Recall measures **coverage**. MRR emphasises the **first useful answer**. nDCG measures
the **quality of the ranked list**.

Together they give a more complete view than any single number.

In [12]:
def recall_at_k(
    ranked_ids,
    relevant_ids,
    k
):
    if not relevant_ids:
        return 0.0

    retrieved = set(
        ranked_ids[:k]
    )

    return (
        len(retrieved & relevant_ids)
        / len(relevant_ids)
    )


def reciprocal_rank_at_k(
    ranked_ids,
    relevant_ids,
    k
):
    for rank, passage_id in enumerate(
        ranked_ids[:k],
        start=1
    ):
        if passage_id in relevant_ids:
            return 1.0 / rank

    return 0.0


def ndcg_at_k(
    ranked_ids,
    relevant_ids,
    k
):
    dcg = 0.0

    for rank, passage_id in enumerate(
        ranked_ids[:k],
        start=1
    ):
        relevance = (
            1.0
            if passage_id in relevant_ids
            else 0.0
        )

        if relevance:
            dcg += (
                relevance
                / math.log2(rank + 1)
            )

    ideal_hits = min(
        len(relevant_ids),
        k
    )

    if ideal_hits == 0:
        return 0.0

    idcg = sum(
        1.0 / math.log2(rank + 1)
        for rank in range(
            1,
            ideal_hits + 1
        )
    )

    return dcg / idcg
# All metric functions receive a ranked list plus the ground-truth relevant IDs.
# Keeping metrics independent from the embedding model makes the evaluation
# logic reusable for the fine-tuned model later.


## 11. Retrieve the top-K passages

This function implements the complete dense-retrieval experiment.

### Step 1 — Encode queries

Each query is transformed into a normalised embedding using the same BGE-M3 model used
for passages.

### Step 2 — Calculate similarities

With:

```text
Q = query embedding matrix
P = passage embedding matrix
```

the code computes:

```text
scores = Q × Pᵀ
```

Each row then contains the similarity between one query and **every passage**.

### Step 3 — Select top K

`np.argpartition` efficiently finds the K strongest candidates without fully sorting
the entire corpus. Those candidates are then sorted by similarity.

### Step 4 — Compare with ground truth

The ranked passage IDs are compared with the qrels to calculate Recall, reciprocal
rank and nDCG.

### Step 5 — Keep diagnostic information

The function stores both ranked passage IDs and similarity scores. This allows us to
inspect later *why* a query failed instead of seeing only an aggregate metric.

In [13]:
def retrieve_split(
    queries,
    relevance_map,
    top_k=TOP_K
):
    query_texts = [
        query["query"]
        for query in queries
    ]

    query_embeddings = model.encode(
        query_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    query_embeddings = np.asarray(
        query_embeddings,
        dtype=np.float32
    )

    scores = (
        query_embeddings
        @ passage_embeddings.T
    )

    results = []

    for index, query in enumerate(queries):
        row = scores[index]

        candidate_count = min(
            top_k,
            len(passages)
        )

        top_indices = np.argpartition(
            -row,
            candidate_count - 1
        )[:candidate_count]

        top_indices = top_indices[
            np.argsort(
                -row[top_indices]
            )
        ]

        ranked_ids = [
            passages[i]["passage_id"]
            for i in top_indices
        ]

        ranked_scores = [
            float(row[i])
            for i in top_indices
        ]

        relevant_ids = relevance_map[
            query["query_id"]
        ]

        result = {
            "query_id": query["query_id"],
            "split": query["split"],
            "query_type": query["query_type"],
            "query": query["query"],
            "document_id": query["document_id"],
            "article": query["article"],
            "source_article_id": (
                query["source_article_id"]
            ),
            "relevant_passage_count": (
                len(relevant_ids)
            ),
            "recall_at_1": recall_at_k(
                ranked_ids,
                relevant_ids,
                1
            ),
            "recall_at_5": recall_at_k(
                ranked_ids,
                relevant_ids,
                5
            ),
            "recall_at_10": recall_at_k(
                ranked_ids,
                relevant_ids,
                10
            ),
            "mrr_at_10": reciprocal_rank_at_k(
                ranked_ids,
                relevant_ids,
                10
            ),
            "ndcg_at_10": ndcg_at_k(
                ranked_ids,
                relevant_ids,
                10
            ),
            "top_passage_ids": ranked_ids,
            "top_scores": ranked_scores,
        }

        results.append(result)

    return results
# Because both query and passage embeddings are L2-normalised, this matrix
# multiplication computes cosine-similarity scores efficiently for all
# query-passage combinations.


## 12. Run validation and test retrieval

We now execute the same retrieval procedure independently on validation and test.

The model, passage corpus, similarity function, `TOP_K` and metrics remain identical.
This consistency is necessary for meaningful comparison between splits and, later,
between the baseline and fine-tuned model.

Execution time is recorded because computational cost is also useful experimental
information.

In [14]:
results_by_split = {}

for split in EVALUATION_SPLITS:
    print(
        f"\nEvaluating {split}..."
    )

    start = time.time()

    results_by_split[split] = (
        retrieve_split(
            queries_by_split[split],
            relevance_by_split[split],
            top_k=TOP_K
        )
    )

    print(
        f"Completed in "
        f"{time.time() - start:.1f} s"
    )


Evaluating validation...


Batches: 100%|██████████| 17/17 [00:33<00:00,  1.95s/it]


Completed in 33.2 s

Evaluating test...


Batches: 100%|██████████| 18/18 [00:37<00:00,  2.11s/it]

Completed in 38.0 s


## 13. Aggregate metrics

The previous stage produced metrics **per query**. We now calculate mean performance
across groups of queries.

For each split we report:

```text
all        → every query
semantic   → topic-based queries
reference  → queries containing explicit legal references
```

This separation is methodologically important.

A model can perform very well on:

```text
"O que estabelece o artigo 6.º do CIVA?"
```

by exploiting lexical cues such as `artigo 6.º` and `CIVA`.

A semantic query such as:

```text
"Quais são as regras sobre localização das operações?"
```

is a stronger test of semantic retrieval.

For this project, improvements on the semantic subset are therefore particularly
important.

In [15]:
METRIC_COLUMNS = [
    "recall_at_1",
    "recall_at_5",
    "recall_at_10",
    "mrr_at_10",
    "ndcg_at_10",
]


def aggregate_metrics(records):
    if not records:
        return {
            metric: float("nan")
            for metric in METRIC_COLUMNS
        }

    return {
        metric: float(
            np.mean([
                record[metric]
                for record in records
            ])
        )
        for metric in METRIC_COLUMNS
    }


summary_rows = []

for split, records in results_by_split.items():
    groups = {
        "all": records,
        "semantic": [
            r for r in records
            if r["query_type"] == "semantic"
        ],
        "reference": [
            r for r in records
            if r["query_type"] == "reference"
        ],
    }

    for group_name, group_records in groups.items():
        row = {
            "split": split,
            "query_group": group_name,
            "queries": len(group_records),
            **aggregate_metrics(
                group_records
            ),
        }

        summary_rows.append(row)


summary_df = pd.DataFrame(
    summary_rows
)

summary_df

,split,query_group,queries,recall_at_1,recall_at_5,recall_at_10,mrr_at_10,ndcg_at_10
0,validation,all,516,0.566311,0.786628,0.822190,0.694378,0.723387
1,validation,semantic,387,0.699871,0.937597,0.955426,0.838748,0.866318
2,validation,reference,129,0.165633,0.333721,0.422481,0.261268,0.294594
3,test,all,561,0.512715,0.705333,0.775158,0.690244,0.689190
4,test,semantic,420,0.617652,0.820152,0.879147,0.807162,0.804094
5,test,reference,141,0.200138,0.363318,0.465406,0.341979,0.346922


## 14. Percentage-formatted summary

The underlying metrics are stored numerically between 0 and 1.

For presentation, a value such as:

```text
0.8387
```

can be displayed as:

```text
83.87%
```

This cell changes only the **display format**. It does not alter the values used for
analysis or saved to CSV.

In [16]:
display_df = summary_df.copy()

for metric in METRIC_COLUMNS:
    display_df[metric] = (
        display_df[metric]
        .map(
            lambda value:
            f"{value:.2%}"
            if pd.notna(value)
            else ""
        )
    )

display_df

,split,query_group,queries,recall_at_1,recall_at_5,recall_at_10,mrr_at_10,ndcg_at_10
0,validation,all,516,56.63%,78.66%,82.22%,69.44%,72.34%
1,validation,semantic,387,69.99%,93.76%,95.54%,83.87%,86.63%
2,validation,reference,129,16.56%,33.37%,42.25%,26.13%,29.46%
3,test,all,561,51.27%,70.53%,77.52%,69.02%,68.92%
4,test,semantic,420,61.77%,82.02%,87.91%,80.72%,80.41%
5,test,reference,141,20.01%,36.33%,46.54%,34.20%,34.69%


## 15. Results by legal document

Global averages can hide substantial differences between legal sources.

For example, the model may retrieve CIVA provisions well but struggle with another
code whose vocabulary or structure differs.

We therefore aggregate the same metrics by:

- split;
- legal document;
- query type.

Document-level analysis can reveal where future training data or parser improvements
are most needed.

In [17]:
document_rows = []

for split, records in results_by_split.items():
    documents = sorted(
        set(
            r["document_id"]
            for r in records
        )
    )

    for document_id in documents:
        for query_group in [
            "all",
            "semantic",
            "reference"
        ]:
            subset = [
                r
                for r in records
                if (
                    r["document_id"]
                    == document_id
                    and (
                        query_group == "all"
                        or r["query_type"]
                        == query_group
                    )
                )
            ]

            if not subset:
                continue

            document_rows.append({
                "split": split,
                "document_id": document_id,
                "query_group": query_group,
                "queries": len(subset),
                **aggregate_metrics(subset),
            })


document_df = pd.DataFrame(
    document_rows
)

document_df

,split,document_id,query_group,queries,recall_at_1,recall_at_5,recall_at_10,mrr_at_10,ndcg_at_10
0,validation,cimi,all,60,0.533333,0.750000,0.800000,0.618677,0.662311
1,validation,cimi,semantic,45,0.688889,0.911111,0.933333,0.771111,0.810519
2,validation,cimi,reference,15,0.066667,0.266667,0.400000,0.161376,0.217686
3,validation,circ,all,64,0.472656,0.774740,0.821615,0.676215,0.702457
4,validation,circ,semantic,48,0.567708,0.906250,0.932292,0.797106,0.823705
5,validation,circ,reference,16,0.187500,0.380208,0.489583,0.313542,0.338713
6,validation,cirs,all,76,0.500000,0.723684,0.763158,0.612202,0.649304
7,validation,cirs,semantic,57,0.631579,0.912281,0.947368,0.769904,0.813734
8,validation,cirs,reference,19,0.105263,0.157895,0.210526,0.139098,0.156014
9,validation,cis,all,28,0.750000,0.785714,0.821429,0.764881,0.778103


## 16. Semantic test results by document

This view focuses specifically on semantic queries from the held-out test set.

Sorting documents by retrieval performance helps identify which legal sources are
currently easiest or hardest for the original BGE-M3 model.

These results should be interpreted together with the number of queries per document:
a metric based on very few queries is less stable than one based on many.

In [18]:
semantic_test_by_document = (
    document_df[
        (document_df["split"] == "test")
        & (
            document_df["query_group"]
            == "semantic"
        )
    ]
    .sort_values(
        "recall_at_5",
        ascending=False
    )
    .reset_index(drop=True)
)

semantic_test_by_document

,split,document_id,query_group,queries,recall_at_1,recall_at_5,recall_at_10,mrr_at_10,ndcg_at_10
0,test,oe2026,semantic,78,0.879487,1.000000,1.000000,0.960897,0.970645
1,test,riti,semantic,12,0.750000,0.958333,1.000000,0.902778,0.929659
2,test,rgit,semantic,36,0.770833,0.951389,1.000000,0.939815,0.946082
3,test,cimi,semantic,48,0.562500,0.895833,0.937500,0.734458,0.778304
4,test,lgt,semantic,39,0.634615,0.884615,0.910256,0.825641,0.830211
5,test,ebf,semantic,39,0.583333,0.839744,0.858974,0.829060,0.821806
6,test,cis,semantic,24,0.645833,0.781250,0.854167,0.774306,0.786325
7,test,circ,semantic,54,0.372016,0.653807,0.754012,0.789095,0.703700
8,test,civa,semantic,30,0.466667,0.627778,0.750000,0.686111,0.647946
9,test,cirs,semantic,60,0.499583,0.626250,0.758750,0.630437,0.644951


## 17. Inspect retrieval failures

Aggregate metrics tell us **how often** retrieval succeeds. Failure analysis helps us
understand **why** it fails.

For one query, `inspect_result()` displays:

- the query;
- expected relevant passage IDs;
- retrieved passages;
- similarity scores;
- whether each retrieved passage is relevant;
- the actual legal text.

A failure may reveal different problems:

```text
Model problem
→ semantically similar but wrong provision ranked first

Dataset problem
→ query is ambiguous

Parsing problem
→ relevant article text is malformed

Chunking problem
→ useful context was split poorly
```

Not every low metric should automatically be blamed on the embedding model.

In [19]:
passage_by_id = {
    passage["passage_id"]: passage
    for passage in passages
}


def inspect_result(
    result,
    max_results=5,
    max_chars=1000
):
    relevant = relevance_by_split[
        result["split"]
    ][result["query_id"]]

    print("=" * 90)
    print("QUERY")
    print(result["query"])
    print(
        f"Type: {result['query_type']} | "
        f"Document: {result['document_id'].upper()} | "
        f"Article: {result['article']}"
    )

    print("\nRelevant passage IDs:")
    for pid in sorted(relevant):
        print(" -", pid)

    print("\nRetrieved:")

    for rank, (
        passage_id,
        score
    ) in enumerate(
        zip(
            result["top_passage_ids"],
            result["top_scores"]
        ),
        start=1
    ):
        passage = passage_by_id[
            passage_id
        ]

        marker = (
            "✓"
            if passage_id in relevant
            else " "
        )

        print(
            f"\n[{rank:02d}] {marker} "
            f"score={score:.4f} | "
            f"{passage_id}"
        )

        print(
            passage["text"][:max_chars]
        )

## 18. Worst semantic queries on the test set

We sort semantic test queries from weakest to strongest retrieval performance.

Queries with `Recall@10 = 0` are especially informative because none of their expected
passages appears among the ten highest-scoring results.

These cases are valuable candidates for later **hard-negative mining**: the passages
incorrectly ranked above the target may be precisely the confusing examples the model
needs to learn from.

In [20]:
semantic_test_results = [
    result
    for result in results_by_split["test"]
    if result["query_type"] == "semantic"
]

worst_semantic = sorted(
    semantic_test_results,
    key=lambda r: (
        r["recall_at_10"],
        r["mrr_at_10"],
        r["ndcg_at_10"],
    )
)

print(
    "Semantic test queries with "
    "Recall@10 = 0:",
    sum(
        r["recall_at_10"] == 0
        for r in semantic_test_results
    )
)

for result in worst_semantic[:10]:
    print(
        f"{result['document_id'].upper():8} | "
        f"Art. {result['article']:8} | "
        f"R@10={result['recall_at_10']:.2f} | "
        f"{result['query']}"
    )

Semantic test queries with Recall@10 = 0: 38
CIMI     | Art. 88.º     | R@10=0.00 | Quais são as regras sobre Publicação?
CIMI     | Art. 88.º     | R@10=0.00 | Como é regulada a matéria de Publicação no CIMI?
CIMI     | Art. 88.º     | R@10=0.00 | O que prevê o CIMI relativamente a Publicação?
CIRC     | Art. 121.º-B  | R@10=0.00 | Quais são as regras sobre (Aditado pela Lei n.o 98/2017, de 24 de agosto)?
CIRC     | Art. 121.º-B  | R@10=0.00 | Como é regulada a matéria de (Aditado pela Lei n.o 98/2017, de 24 de agosto) no CIRC?
CIRC     | Art. 121.º-B  | R@10=0.00 | O que prevê o CIRC relativamente a (Aditado pela Lei n.o 98/2017, de 24 de agosto)?
CIRC     | Art. 143.º    | R@10=0.00 | Quais são as regras sobre (Aditado pela Lei n.o 119/2019, de 18 de setembro)?
CIRC     | Art. 143.º    | R@10=0.00 | Como é regulada a matéria de (Aditado pela Lei n.o 119/2019, de 18 de setembro) no CIRC?
CIRC     | Art. 143.º    | R@10=0.00 | O que prevê o CIRC relativamente a (Aditado pela Lei n.o 1

## 19. Inspect one failure in detail

If at least one semantic query has zero Recall@10, we inspect the weakest example.

During a training session, compare:

1. the wording of the query;
2. the expected legal provision;
3. the top retrieved passages;
4. their similarity scores.

Ask whether the retrieved passages are genuinely irrelevant or whether the qrels/query
generation need refinement.

This manual analysis connects quantitative evaluation with qualitative judgement.

In [21]:
failed = [
    result
    for result in worst_semantic
    if result["recall_at_10"] == 0
]

if failed:
    inspect_result(
        failed[0],
        max_results=5
    )
else:
    print(
        "No semantic test query failed "
        "at Recall@10."
    )

QUERY
Quais são as regras sobre Publicação?
Type: semantic | Document: CIMI | Article: 88.º

Relevant passage IDs:
 - cimi_block_02_art_88_o

Retrieved:

[01]   score=0.5453 | cis_block_02_art_65_o
Documento: Código do Imposto do Selo
Estrutura: Chapter IX — Disposições Diversas
Artigo 65.º — Letras e livranças
1 - As letras emitidas obedecerão aos requisitos previstos na lei uniforme relativa a
letras e livranças.
2 - O modelo das letras e livranças e suas características são estabelecidos em
portaria do Ministro das Finanças.
3 - As letras são editadas oficialmente ou, facultativamente, pelas empresas públicas
e sociedades regularmente constituídas, desde que o número de letras emitidas
durante o ano não seja inferior a 600.
4 - Para efeitos da segunda parte do número anterior, podem as entidades nele
referidas emitir letras no ano de início da sua actividade quando prevejam que o
número de letras a emitir nesse ano será igual ou superior ao múltiplo do número de
meses de calendário 

## 20. Save detailed retrieval results

Per-query results are exported to JSONL.

This preserves information that would be lost in aggregate CSV tables, including:

- each original query;
- its metrics;
- ranked passage IDs;
- similarity scores.

Detailed outputs make later error analysis possible without rerunning the entire
embedding model.

In [22]:
def json_safe_result(result):
    return {
        **result,
        "top_scores": [
            float(score)
            for score in result[
                "top_scores"
            ]
        ],
    }


for split, records in results_by_split.items():
    path = (
        RESULTS_DIR
        / f"{split}_retrieval_results.jsonl"
    )

    with path.open(
        "w",
        encoding="utf-8"
    ) as f:
        for record in records:
            f.write(
                json.dumps(
                    json_safe_result(record),
                    ensure_ascii=False
                )
                + "\n"
            )

    print(path.resolve())

C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\baseline\validation_retrieval_results.jsonl
C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\baseline\test_retrieval_results.jsonl


## 21. Save metric tables

Two CSV files summarise the baseline:

```text
baseline_summary.csv
    overall + semantic + reference metrics

baseline_by_document.csv
    metrics broken down by legal document
```

CSV is convenient here because the results are tabular and can be inspected in
spreadsheets, Python, R or BI tools.

In [23]:
SUMMARY_PATH = (
    RESULTS_DIR
    / "baseline_summary.csv"
)

DOCUMENT_PATH = (
    RESULTS_DIR
    / "baseline_by_document.csv"
)

summary_df.to_csv(
    SUMMARY_PATH,
    index=False
)

document_df.to_csv(
    DOCUMENT_PATH,
    index=False
)

print(SUMMARY_PATH.resolve())
print(DOCUMENT_PATH.resolve())

C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\baseline\baseline_summary.csv
C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\baseline\baseline_by_document.csv


## 22. Save baseline metadata

Metrics are only reproducible when we also know the configuration that produced them.

The metadata records information such as:

- model name;
- device;
- embedding dimension;
- maximum sequence length;
- batch size;
- top-K;
- number of passages;
- number of queries and qrels.

Later, the fine-tuned evaluation should use compatible settings so differences can be
attributed to model adaptation rather than to a changed evaluation protocol.

In [24]:
metadata = {
    "evaluation_version": "1.0",
    "model_name": MODEL_NAME,
    "device": DEVICE,
    "embedding_dimension": (
        model.get_sentence_embedding_dimension()
    ),
    "max_seq_length": MAX_SEQ_LENGTH,
    "batch_size": BATCH_SIZE,
    "top_k": TOP_K,
    "passages": len(passages),
    "splits": {
        split: {
            "queries": len(
                queries_by_split[split]
            ),
            "qrels": len(
                qrels_by_split[split]
            ),
        }
        for split in EVALUATION_SPLITS
    },
}

METADATA_PATH = (
    RESULTS_DIR
    / "baseline_metadata.json"
)

with METADATA_PATH.open(
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2
    )

print(METADATA_PATH.resolve())
# Saving the evaluation configuration alongside the scores prevents later
# comparisons from accidentally mixing results produced with different settings.


C:\Users\user\Documents\GitHub\legal-embeddings-finetuning-pt\results\baseline\baseline_metadata.json


C:\Users\user\AppData\Local\Temp\ipykernel_23704\3904177988.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


## 23. Baseline interpretation checklist

Before moving to fine-tuning, answer these questions.

### 1. Are reference queries much easier than semantic queries?

If yes, the model is benefiting strongly from explicit article/document identifiers.
This is expected, but it reinforces the importance of semantic evaluation.

### 2. Which metric matters most for the intended application?

For a RAG system, obtaining at least one strong relevant passage near the top is often
important. MRR can therefore be particularly informative, while Recall remains
important when an answer depends on several chunks.

### 3. Which documents perform worst?

This may indicate vocabulary differences, corpus imbalance or document-specific
parsing issues.

### 4. What do failed queries retrieve instead?

Semantically plausible false positives are useful training signals. Completely
unrelated results may indicate a more fundamental representation problem.

### 5. Is the evaluation dataset itself credible?

Synthetic queries and automatically derived qrels should always be inspected. A metric
is only meaningful if its ground truth is meaningful.

The baseline numbers saved here become the reference point for the rest of the
experiment.

## 24. Next stage — hard-negative mining

The baseline model has now done more than give us a score. It has shown us **which
incorrect passages it finds confusing**.

That information can be used to construct stronger training examples.

Consider a semantic query:

```text
Query
"Quais são as regras sobre localização das operações?"
```

Suppose BGE-M3 produces:

```text
Rank 1 → wrong but semantically similar article
Rank 2 → another wrong related article
Rank 3 → correct article
```

The passages at ranks 1 and 2 are much more useful negatives than a random unrelated
article.

These are **hard negatives**.

```text
                    BASELINE MODEL
                         │
Query ──────────────────►│
                         │
                         ▼
                  ranked passages
                         │
             ┌───────────┴───────────┐
             ▼                       ▼
        relevant passage       high-ranked wrong
                                  passage
                                      │
                                      ▼
                               HARD NEGATIVE
```

The next stage will therefore:

1. use the baseline model to retrieve candidate passages;
2. remove all passages belonging to the relevant article;
3. retain high-scoring non-relevant passages;
4. create stronger `(query, positive, hard negative)` training examples.

After fine-tuning, the adapted model must be evaluated against this **same baseline
protocol**.

Only then can we answer the central experimental question:

> **Did domain-specific fine-tuning improve Portuguese legal semantic retrieval?**